# Objective :Reward oracle for bandits simulator





Given: Features ↓ Incremental delivery duration

In [4]:
local = True

# Configure SIMD before importing NumPy/SciPy when running locally on AVX-512 hardware.
if local:
    import os
    os.environ.setdefault('MKL_ENABLE_INSTRUCTIONS', 'AVX512')
    os.environ.setdefault('OMP_NUM_THREADS', '1')  # joblib owns parallelism
    os.environ.setdefault('OPENBLAS_CORETYPE', 'SKYLAKEX')
    print('Local AVX-512 hints set (MKL_ENABLE_INSTRUCTIONS=AVX512, OPENBLAS_CORETYPE=SKYLAKEX)')

if not local:
    from google.colab import drive
    drive.mount('/content/drive')

Local AVX-512 hints set (MKL_ENABLE_INSTRUCTIONS=AVX512, OPENBLAS_CORETYPE=SKYLAKEX)


In [8]:
! pip install lightgbm scikit-learn scikit-learn-intelex

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 10.3 MB/s  0:00:00
   ---------------------------------------- 0.0/25.3 MB ? eta -:--:--
   --- ------------------------------------ 2.4/25.3 MB 11.8 MB/s eta 0:00:02
   ------- -------------------------------- 5.0/25.3 MB 11.9 MB/s eta 0:00:02
   --------- ------------------------------ 6.3/25.3 MB 11.9 MB/s eta 0:00:02
   --------- ------------------------------ 6.3/25.3 MB 11.9 MB/s eta 0:00:02
   --------- ------------------------------ 6.3/25.3 MB 11.9 MB/s eta 0:00:02
   --------- ------------------------------ 6.3/25.3 MB 11.9 MB/s eta 0:00:02
   ------------ --------------------------- 7.6/25.3 MB 5.2 MB/s eta 0:00:04
   --------------- ------------------------ 10.0/25.3 MB 5.8 MB/s eta 0:00:03
   ------------------- -------------------- 12.6/25.3 MB 6.6 MB/s eta 0:00:02
   ----------------------- ---------------- 14.9/25.3 MB 7.0 MB/s eta 0:00:02
   -----

In [ ]:
if local :
# 1. Apply the AMD acceleration patch FIRST
    # 1. Trigger the AVX-512 level optimizations
    from sklearnex import patch_sklearn
    patch_sklearn()

    

Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


In [13]:
from pathlib import Path
if local:
    root = Path('data')
    if not root.exists():
        root = Path('.data')
else:
    root = Path('/content/drive/MyDrive/ml/CORRECTEDv3')

DATA_PATHS = {
    'Chongqing': root / 'delivery_features_chongqing.parquet',
    'Shanghai': root / 'delivery_features_shanghai.parquet',
    'Hangzhou': root / 'delivery_features_hangzhou.parquet',
}

OUTPUT_DIR = root / 'causal_graphs' / 'pc_delivery'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
STATIC_FEATURES = [

    # Structural
    "pickup_destination_distance",
    "batch_size",
    "batch_rank_dispatch",
    "same_aoi_share_in_batch",
    "isolated_delivery",
    "distance_to_batch_centroid",
    "typecode_cb",

    # Operational
    "courier_eta_ewm",
    "gps_points",
    "speed_mean_15m",
    "speed_std_15m",
    "distance_travelled_15m",
    "coverage_ratio",
    "gps_gap_min",
    "idle_fraction",
    "is_trajectory_available",

    # Environment
    "WSI",
    "temperature_2m",
    "precipitation",
    "windspeed_10m",
    "spatial_congestion_daily",
    "spatial_congestion_norm",

    # Time
    "hour_sin",
    "hour_cos",
    "is_weekend",
    "is_holiday",
    "is_holiday_eve"
]




In [ ]:
DYNAMIC_FEATURES = [

    "dist_from_current",
    "remaining_orders",
    "batch_progress",
    "elapsed_route_time",
    "last_duration",
    "current_hour",
    "cumulative_distance"
]